In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, RandomizedSearchCV, cross_val_predict
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
import warnings
warnings.filterwarnings("ignore")
sns.set(style="whitegrid")

In [ ]:
# Final Flight Difficulty Score pipeline
# - Assumes merged_df (and optionally flight_final) are already loaded in the notebook
# - Produces: interpretable_score (Ridge), ml_difficulty_score (RandomForest)
# - Per-day normalization (scores reset each day), daily ranks, and 3-class labels (Easy/Medium/Difficult)
# - Saves results and trained RF model (pickle)
# ---------------------------------------------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupKFold, RandomizedSearchCV, cross_val_predict
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import warnings
warnings.filterwarnings("ignore")
sns.set(style="whitegrid")
RANDOM_STATE = 42

# -------------------------
# Prepare date grouping column
# -------------------------
merged_df['dep_date'] = pd.to_datetime(merged_df['scheduled_departure_date_local']).dt.date

# -------------------------
# Feature lists (use columns present in merged_df)
# -------------------------
numeric_features = [
    'load_factor', 'total_passengers', 'bags_per_passenger', 'total_bags',
    'num_hot_transfer_bags', 'num_transfer_bags', 'scheduled_ground_time_minutes',
    'actual_ground_time_minutes', 'minimum_turn_minutes', 'ground_time_ratio',
    'ssr_per_passenger', 'child_passenger_ratio', 'basic_economy_ratio',
    'avg_pax_per_pnr', 'stroller_ratio'
]
numeric_features = [c for c in numeric_features if c in merged_df.columns]

categorical_features = [c for c in ['fleet_type','carrier','scheduled_departure_station_code',
                                    'scheduled_arrival_station_code','company_id']
                        if c in merged_df.columns]

print(f"Using numeric features: {numeric_features}")
print(f"Using categorical features: {categorical_features}")

# -------------------------
# Prepare dataframe for modeling
# -------------------------
df = merged_df.copy()

# Target
if 'delay_minutes' not in df.columns:
    df['delay_minutes'] = (pd.to_datetime(df['actual_departure_datetime_local']) -
                           pd.to_datetime(df['scheduled_departure_datetime_local'])).dt.total_seconds() / 60.0

# Fill missing numeric values with median (simple, effective)
for c in numeric_features:
    df[c] = df[c].fillna(df[c].median())

# Fill missing categorical
for c in categorical_features:
    df[c] = df[c].fillna("UNKNOWN")

# X, y, groups
X = df[numeric_features + categorical_features].copy()
y = df['delay_minutes'].values
groups = df['dep_date'].values

# -------------------------
# Interpretable additive score: per-day min-max normalize numeric features, learn weights via Ridge
# -------------------------
# Normalize numeric features per day (min-max) so contributions are comparable within each day
X_ridge = df[numeric_features].copy()
for col in numeric_features:
    X_ridge[col] = df.groupby('dep_date')[col].transform(lambda s: (s - s.min()) / (s.max() - s.min() + 1e-9))

# Fit Ridge (interpretable linear weights)
ridge = Ridge(alpha=1.0, random_state=RANDOM_STATE)
ridge.fit(X_ridge, df['delay_minutes'])
ridge_coefs = pd.Series(ridge.coef_, index=numeric_features).sort_values(key=abs, ascending=False)
print("\nTop Ridge coefficients (absolute):")
print(ridge_coefs.head(15))

# Compute interpretable score (weighted sum), then daily min-max normalize to [0,1]
df['interpretable_raw'] = X_ridge.values.dot(ridge.coef_) + ridge.intercept_
df['interpretable_score'] = df.groupby('dep_date')['interpretable_raw'].transform(lambda g: (g - g.min())/(g.max() - g.min() + 1e-9))
df['interpretable_rank'] = df.groupby('dep_date')['interpretable_score'].rank(method='dense', ascending=False).astype(int)
# Daily tertiles -> Easy/Medium/Difficult
def per_day_tertile(series):
    if series.nunique() < 3:
        return pd.Series(['Easy'] * len(series), index=series.index)
    return pd.qcut(series, q=3, labels=['Easy','Medium','Difficult'])
df['interpretable_class'] = df.groupby('dep_date')['interpretable_score'].apply(per_day_tertile).reset_index(level=0, drop=True)

# -------------------------
# ML-based score: RandomForest with grouped CV and randomized hyperparam search
# -------------------------
# Preprocessor
num_transform = ('num', StandardScaler(), numeric_features)
cat_transform = ('cat', OneHotEncoder(handle_unknown='ignore', sparse=False), categorical_features)
preprocessor = ColumnTransformer(transformers=[num_transform, cat_transform], remainder='drop')

rf = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)
pipe = Pipeline([('pre', preprocessor), ('rf', rf)])

param_dist = {
    'rf__n_estimators': [100, 200, 300],
    'rf__max_depth': [6, 10, 15, None],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 4]
}

gkf = GroupKFold(n_splits=5)
rs = RandomizedSearchCV(pipe, param_dist, n_iter=12,
                        cv=gkf.split(X, y, groups=groups),
                        scoring='neg_root_mean_squared_error',
                        n_jobs=-1, random_state=RANDOM_STATE, verbose=1)

print("\nRunning RandomizedSearchCV for RandomForest (grouped-by-date CV)...")
rs.fit(X, y)
print("Best RF params:", rs.best_params_)

best_rf = rs.best_estimator_

# Cross-validated predictions using grouped splits
print("\nCross-validated predictions (GroupKFold)...")
y_pred_cv = cross_val_predict(best_rf, X, y, cv=gkf.split(X, y, groups=groups), n_jobs=-1)
rmse = mean_squared_error(y, y_pred_cv, squared=False)
mae = mean_absolute_error(y, y_pred_cv)
r2 = r2_score(y, y_pred_cv)
print(f"RF CV results: RMSE={rmse:.3f}, MAE={mae:.3f}, R2={r2:.3f}")

# Fit on full dataset and create ML-based difficulty score
best_rf.fit(X, y)
df['ml_predicted_delay'] = best_rf.predict(X)
df['ml_difficulty_score'] = df.groupby('dep_date')['ml_predicted_delay'].transform(lambda g: (g - g.min())/(g.max() - g.min() + 1e-9))
df['ml_daily_rank'] = df.groupby('dep_date')['ml_difficulty_score'].rank(method='dense', ascending=False).astype(int)
df['ml_difficulty_class'] = df.groupby('dep_date')['ml_difficulty_score'].apply(per_day_tertile).reset_index(level=0, drop=True)

# -------------------------
# Feature importances from RF (top 20)
# -------------------------
# Extract feature names (numeric + OHE features)
num_names = numeric_features
ohe = best_rf.named_steps['pre'].named_transformers_['cat']
try:
    cat_names = list(ohe.get_feature_names_out(categorical_features))
except Exception:
    # fallback
    cat_names = categorical_features
feature_names = num_names + cat_names
importances = best_rf.named_steps['rf'].feature_importances_
fi = pd.Series(importances, index=feature_names).sort_values(ascending=False)
print("\nTop 15 RF feature importances:")
print(fi.head(15))

# -------------------------
# Save outputs and the model
# -------------------------
df_out = df.copy()
df_out.to_csv("flight_with_difficulty_scores.csv", index=False)
joblib.dump(best_rf, "flight_difficulty_rf_model.joblib")
print("\nSaved 'flight_with_difficulty_scores.csv' and 'flight_difficulty_rf_model.joblib'")

# -------------------------
# Quick checks / descriptive summaries
# -------------------------
print("\nMedian delay by ML difficulty class:")
print(df_out.groupby('ml_difficulty_class')['delay_minutes'].median().reindex(['Easy','Medium','Difficult']))

sample_date = df_out['dep_date'].mode()[0]
print(f"\nSample top-10 difficult flights for {sample_date} (by ML score):")
display_cols = ['company_id','flight_number','dep_date','ml_difficulty_score','ml_daily_rank','ml_difficulty_class','delay_minutes','load_factor','total_passengers']
print(df_out[df_out['dep_date']==sample_date][display_cols].sort_values('ml_difficulty_score', ascending=False).head(10).to_string(index=False))

# -------------------------
# Plots: (each chart separate)
# -------------------------
# 1) Distribution of ML difficulty scores (daily-normalized)
plt.figure(figsize=(8,4))
plt.hist(df_out['ml_difficulty_score'], bins=30)
plt.title("Distribution of ML Difficulty Scores (daily-normalized)")
plt.xlabel("Difficulty score (0-1)")
plt.ylabel("Count")
plt.show()

# 2) Median actual delay by difficulty class (ML)
class_med = df_out.groupby('ml_difficulty_class')['delay_minutes'].median().reindex(['Easy','Medium','Difficult'])
plt.figure(figsize=(6,4))
plt.bar(class_med.index.astype(str), class_med.values)
plt.title("Median Delay by ML Difficulty Class")
plt.xlabel("ML Difficulty Class")
plt.ylabel("Median Delay (minutes)")
plt.show()

# 3) Boxplot: delay distribution per class (ML)
plt.figure(figsize=(8,5))
plt.boxplot([df_out[df_out['ml_difficulty_class']=='Easy']['delay_minutes'],
             df_out[df_out['ml_difficulty_class']=='Medium']['delay_minutes'],
             df_out[df_out['ml_difficulty_class']=='Difficult']['delay_minutes']],
            labels=['Easy','Medium','Difficult'])
plt.title("Delay distribution by ML Difficulty Class")
plt.ylabel("Delay (minutes)")
plt.show()

# 4) Top feature importances bar chart
top20 = fi.head(20)[::-1]
plt.figure(figsize=(8,6))
plt.barh(top20.index, top20.values)
plt.title("Top 20 Feature Importances (RF)")
plt.xlabel("Importance")
plt.show()

# -------------------------
# END - df_out contains the final scored table with:
#   - interpretable_score, interpretable_class, interpretable_rank
#   - ml_difficulty_score, ml_difficulty_class, ml_daily_rank
#   - other original columns
# -------------------------